In [ ]:
import pandas as pd
from shared_config.sqlserver import FCC_MATERIAL_DATABASE, PROD_SERVER
from shared_utils.sqlserver import get_engine, replace_table
from sqlalchemy import create_engine
import sys

In [ ]:
SAP_LOT_DATA_PATH = r'C:\analytics-work\production\extract-material-cert-fcc\data\AS - ASBU FCC Engineering - CertDataFiles\SAP_Lot_Data.xlsx'

In [ ]:
def build_lot_summary(df, material_info):
    """Per-(material, vendor_lot) average CCPT vs CPT targets."""
    if df.empty or df['ccpt'].isna().all():
        return pd.DataFrame()

    g = (
        df.dropna(subset=['ccpt'])
        .groupby(['material', 'vendor_lot'], as_index=False)
        .agg(ccpt_lot_avg=('ccpt', 'mean'), n_sublots=('ccpt', 'size'))
    )
    g['ccpt_lot_avg'] = g['ccpt_lot_avg'].round(8)

    pi_indexed = material_info.set_index('material')
    g['cpt_target_pos'] = g['material'].map(
        lambda m: pi_indexed.loc[m, 'cpt_target_pos'] if m in pi_indexed.index else None
    )
    g['cpt_target_neg'] = g['material'].map(
        lambda m: pi_indexed.loc[m, 'cpt_target_neg'] if m in pi_indexed.index else None
    )
    g['in_range'] = g.apply(
        lambda r: (
            None
            if pd.isna(r['cpt_target_pos'])
            else bool(r['cpt_target_neg'] <= r['ccpt_lot_avg'] <= r['cpt_target_pos'])
        ),
        axis=1,
    )
    return g

In [ ]:
def get_oracle_engine():
    """
    SQLAlchemy engine for the SFMFG Oracle database (Solumina), THIN mode.

    Uses:
      - Host: exa4-scan
      - Port: 1521
      - Service name: prp1crpt.orbitalatk.com
      - User: engro
      - Password: sfmfgSoluminaG8

    NOTE: Hardcoding credentials is convenient for testing but unsafe for
    production code. Prefer environment variables or a secrets manager.
    """

    host = 'exa4-scan'
    port = 1521
    service_name = 'prp1crpt.orbitalatk.com'
    user = 'engro'
    password = 'sfmfgSoluminaG8'

    url = f'oracle+oracledb://{user}:{password}@{host}:{port}/?service_name={service_name}'
    return create_engine(url)

In [ ]:
# SFMFG query -- one row per roll/spool used on a Cut Comp Plies work order
SFMFG_QUERY = """
SELECT DISTINCT
    o.PART_NO,
    s.SERIAL_NO,
    o.ORDER_NO,
    op.OPER_NO,
    op.ACTUAL_END_DATE,
    op.OPER_STATUS,
    o.ACTUAL_END_DATE AS WOCLOSE,
    CASE
        WHEN op.ACTUAL_END_DATE IS NULL THEN op.TIME_STAMP
        ELSE op.ACTUAL_END_DATE
    END AS CLOSE_DATE,
    op.TITLE,
    bom.PART_NO AS MATPART,
    bom.LOT_NO,
    bom.SPOOL_NO,
    bom.PART_QTY
FROM SFMFG.SFWID_OPER_DESC op
INNER JOIN SFMFG.SFWID_ORDER_DESC o
    ON op.ORDER_ID = o.ORDER_ID
INNER JOIN SFMFG.SFWID_SERIAL_DESC s
    ON o.ORDER_ID = s.ORDER_ID
INNER JOIN SFMFG.SFWID_AS_WORKED_BOM bom
    ON s.ORDER_ID = bom.ORDER_ID
WHERE o.PART_NO LIKE '2WSH%'
  AND (o.ORDER_NO LIKE '00100%' OR o.ORDER_NO LIKE '34F14%')
  AND op.OPER_STATUS NOT IN ('PENDING')
  AND op.TITLE = 'Cut Comp Plies'
  AND bom.PART_NO LIKE '{material}'
ORDER BY CLOSE_DATE
"""


def fetch_sfmfg_data(oracle_engine, material):
    """Pull raw Cut Comp Plies work order / roll data from Solumina (SFMFG)."""
    query = SFMFG_QUERY.format(material=material)
    df = pd.read_sql(query, oracle_engine)
    df.columns = df.columns.str.lower()
    return df

In [ ]:
def load_batch_to_vendor_lot(material, path=SAP_LOT_DATA_PATH):
    sheet_name = f'{material}_Rolls'
    df = pd.read_excel(path, sheet_name=sheet_name)
    df = df[['Batch', 'Supplier Batch']].dropna(subset=['Batch'])
    return dict(zip(df['Batch'], df['Supplier Batch'], strict=True))

In [ ]:
def build_row_level_ccpt(sfmfg_df, batch_to_vendor_lot, lot_summary, material):
    """
    Add vendor_lot, material, and ccpt to each SFMFG row — one row per roll/spool.
    Assumes sfmfg_df is already filtered to this material.
    """
    df = sfmfg_df.copy()

    df['material'] = material
    df['comp_ply'] = (df['oper_status'] != 'EXCLUDE').astype(int)
    df['month_year'] = pd.to_datetime(df['close_date']).dt.strftime('%Y-%m')

    df['vendor_lot'] = df['lot_no'].map(batch_to_vendor_lot)

    ccpt_lookup = lot_summary.set_index('vendor_lot')['ccpt_lot_avg']
    df['ccpt'] = df['vendor_lot'].map(ccpt_lookup)

    missing = df['ccpt'].isna().sum()
    if missing:
        print(
            f'[{material}] WARNING: {missing} roll(s) had no CCPT match '
            f'(unmapped vendor_lot or missing from lot_summary).'
        )

    return df

In [ ]:
MATERIALS = [
    'LMAMA001FR49GRACL1-INCH',
    'LMAMB002FR49ST1-INCH',
]

prod_engine = get_engine(PROD_SERVER, FCC_MATERIAL_DATABASE)
material_info = pd.read_sql('SELECT * FROM dbo.material_info', prod_engine)
full_cert_df = pd.read_sql('SELECT * FROM material_cert_fcc', prod_engine)

oracle_engine = get_oracle_engine()

all_summaries = []

for material in MATERIALS:
    sfmfg_df = fetch_sfmfg_data(oracle_engine, material)

    batch_to_vendor_lot = load_batch_to_vendor_lot(material)

    cert_for_material = full_cert_df[full_cert_df['material'] == material]
    lot_summary = build_lot_summary(cert_for_material, material_info)

    row_df = build_row_level_ccpt(sfmfg_df, batch_to_vendor_lot, lot_summary, material)
    valid_rows = row_df.dropna(subset=['ccpt'])

    monthly_summary = (
        valid_rows.groupby('month_year', as_index=False)
        .agg(
            avg_ccpt=('ccpt', 'mean'),
            comp_ply_rate=('comp_ply', 'mean'),
            n_rows=('order_no', 'count'),
        )
        .assign(material=material)  # add material column
        .sort_values('month_year')
        .reset_index(drop=True)
    )

    all_summaries.append(monthly_summary)

# Combined summary across all materials
combined_monthly_summary = pd.concat(all_summaries, ignore_index=True)

replace_table(combined_monthly_summary, 'monthly_summary_fcc', prod_engine)